# NIFTY100 Financial Analytics

## Sprint 2 – Day 7

# Growth Analytics

This notebook analyzes the historical growth of companies using
financial statements stored in SQLite.

### Objectives

- Connect to SQLite Database
- Calculate Sales Growth
- Calculate Profit Growth
- Calculate CAGR
- Rank Companies
- Export Results

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
ROOT = Path("../")

DATABASE = ROOT / "data" / "db" / "nifty100.db"

conn = sqlite3.connect(DATABASE)

print("Connected Successfully!")

Connected Successfully!


In [3]:
profit = pd.read_sql(
    "SELECT * FROM profitandloss",
    conn
)

companies = pd.read_sql(
    "SELECT * FROM companies",
    conn
)

In [4]:
print("Profit & Loss:", profit.shape)
print("Companies:", companies.shape)

Profit & Loss: (1276, 15)
Companies: (92, 12)


## Sales Growth Analysis

Calculate Year-over-Year (YoY) Sales Growth for every company.

In [5]:
sales_growth = profit[
    [
        "company_id",
        "year",
        "sales"
    ]
].copy()

sales_growth = sales_growth.sort_values(
    ["company_id", "year"]
)

sales_growth["previous_sales"] = (
    sales_growth
    .groupby("company_id")["sales"]
    .shift(1)
)

sales_growth["sales_growth_pct"] = (
    (
        sales_growth["sales"] -
        sales_growth["previous_sales"]
    )
    /
    sales_growth["previous_sales"]
) * 100

sales_growth.head(15)

,company_id,year,sales,previous_sales,sales_growth_pct
0,ABB,2012.0,1653.0,NaN,NaN
1,ABB,2014.0,2276.0,1653.0,37.689050
2,ABB,2015.0,2289.0,2276.0,0.571178
3,ABB,2016.0,2614.0,2289.0,14.198340
4,ABB,2017.0,2903.0,2614.0,11.055853
5,ABB,2018.0,3298.0,2903.0,13.606614
6,ABB,2019.0,3679.0,3298.0,11.552456
7,ABB,2020.0,4093.0,3679.0,11.253058
8,ABB,2021.0,4310.0,4093.0,5.301735
9,ABB,2022.0,4913.0,4310.0,13.990719


In [6]:
OUTPUT = ROOT / "outputs"

OUTPUT.mkdir(exist_ok=True)

sales_growth.to_csv(
    OUTPUT / "sales_growth.csv",
    index=False
)

print("Sales Growth Saved")

Sales Growth Saved


## Net Profit Growth Analysis

This section calculates the Year-over-Year (YoY) Net Profit Growth for every company.

Formula:

Net Profit Growth (%) =
((Current Year Profit - Previous Year Profit) / Previous Year Profit) × 100

In [7]:
profit_growth = profit[
    [
        "company_id",
        "year",
        "net_profit"
    ]
].copy()

profit_growth = profit_growth.sort_values(
    ["company_id", "year"]
)

profit_growth["previous_profit"] = (
    profit_growth
    .groupby("company_id")["net_profit"]
    .shift(1)
)

profit_growth["profit_growth_pct"] = (
    (
        profit_growth["net_profit"] -
        profit_growth["previous_profit"]
    )
    /
    profit_growth["previous_profit"]
) * 100

profit_growth.head(15)

,company_id,year,net_profit,previous_profit,profit_growth_pct
0,ABB,2012.0,145.0,NaN,NaN
1,ABB,2014.0,198.0,145.0,36.551724
2,ABB,2015.0,229.0,198.0,15.656566
3,ABB,2016.0,255.0,229.0,11.353712
4,ABB,2017.0,277.0,255.0,8.627451
5,ABB,2018.0,401.0,277.0,44.765343
6,ABB,2019.0,450.0,401.0,12.219451
7,ABB,2020.0,593.0,450.0,31.777778
8,ABB,2021.0,691.0,593.0,16.526138
9,ABB,2022.0,799.0,691.0,15.629522


In [8]:
profit_growth.to_csv(
    OUTPUT / "profit_growth.csv",
    index=False
)

print("Profit Growth Analysis Saved Successfully!")

Profit Growth Analysis Saved Successfully!


## Compound Annual Growth Rate (CAGR)

CAGR measures the average annual growth rate of a company's sales over multiple years.

Formula:

CAGR = ((Ending Value / Beginning Value) ^ (1 / Number of Years) - 1) × 100

In [9]:
cagr_list = []

for company in profit["company_id"].unique():

    company_data = (
        profit[profit["company_id"] == company]
        .sort_values("year")
    )

    company_data = company_data.dropna(subset=["sales"])

    if len(company_data) < 2:
        continue

    first_sales = company_data.iloc[0]["sales"]
    last_sales = company_data.iloc[-1]["sales"]

    first_year = company_data.iloc[0]["year"]
    last_year = company_data.iloc[-1]["year"]

    years = last_year - first_year

    if years <= 0:
        continue

    if first_sales <= 0 or last_sales <= 0:
        continue

    cagr = (
        ((last_sales / first_sales) ** (1 / years)) - 1
    ) * 100

    cagr_list.append(
        {
            "company_id": company,
            "start_year": first_year,
            "end_year": last_year,
            "sales_cagr_pct": round(cagr, 2)
        }
    )

cagr_analysis = pd.DataFrame(cagr_list)

cagr_analysis.sort_values(
    "sales_cagr_pct",
    ascending=False,
    inplace=True
)

cagr_analysis.head(10)

,company_id,start_year,end_year,sales_cagr_pct
78,SIEMENS,2011.0,2024.0,4.89
0,ABB,2012.0,NaN,NaN
1,ADANIENT,2013.0,NaN,NaN
2,ADANIGREEN,2017.0,NaN,NaN
3,ADANIPORTS,2013.0,NaN,NaN
4,ADANIPOWER,2013.0,NaN,NaN
5,AMBUJACEM,2012.0,NaN,NaN
6,APOLLOHOSP,2013.0,NaN,NaN
7,ASIANPAINT,2013.0,NaN,NaN
8,ATGL,2018.0,NaN,NaN


In [10]:
cagr_analysis.to_csv(
    OUTPUT / "cagr_analysis.csv",
    index=False
)

print("CAGR Analysis Saved Successfully!")

CAGR Analysis Saved Successfully!


## Growth Scorecard

This scorecard combines Sales Growth, Profit Growth, and CAGR into a single growth score for each company.

In [11]:
latest_sales = (
    sales_growth
    .sort_values("year")
    .groupby("company_id")
    .tail(1)
)

latest_profit = (
    profit_growth
    .sort_values("year")
    .groupby("company_id")
    .tail(1)
)

growth_scorecard = latest_sales[
    [
        "company_id",
        "sales_growth_pct"
    ]
].merge(
    latest_profit[
        [
            "company_id",
            "profit_growth_pct"
        ]
    ],
    on="company_id",
    how="outer"
).merge(
    cagr_analysis[
        [
            "company_id",
            "sales_cagr_pct"
        ]
    ],
    on="company_id",
    how="outer"
)

growth_scorecard.fillna(0, inplace=True)

growth_scorecard["growth_score"] = (
    growth_scorecard["sales_growth_pct"] * 0.4 +
    growth_scorecard["profit_growth_pct"] * 0.4 +
    growth_scorecard["sales_cagr_pct"] * 0.2
)

growth_scorecard = growth_scorecard.sort_values(
    by="growth_score",
    ascending=False
)

growth_scorecard.head(10)

,company_id,sales_growth_pct,profit_growth_pct,sales_cagr_pct,growth_score
98,ZOMATO,48.357273,88.888889,0.0,54.898465
96,VEDL,0.501645,110.120706,0.0,44.248940
59,LODHA,31.582009,61.583012,0.0,37.266008
2,ADANIENT,6.108628,82.488756,0.0,35.438953
3,ADANIGREEN,16.941432,53.015873,0.0,27.982922
18,BHEL,8.119533,57.801418,0.0,26.368381
72,PNB,6.474121,58.731025,0.0,26.082059
17,BHARTIARTL,3.664440,61.439589,0.0,26.041611
87,TECHM,0.923148,57.488527,0.0,23.364670
28,DLF,8.931072,46.989721,0.0,22.368317


In [12]:
growth_scorecard.to_csv(
    OUTPUT / "growth_scorecard.csv",
    index=False
)

print("Growth Scorecard Saved Successfully!")

Growth Scorecard Saved Successfully!


## Top Growth Companies

Display the top 20 companies based on the calculated growth score.

In [13]:
top_growth_companies = growth_scorecard.head(20)

top_growth_companies

,company_id,sales_growth_pct,profit_growth_pct,sales_cagr_pct,growth_score
98,ZOMATO,48.357273,88.888889,0.00,54.898465
96,VEDL,0.501645,110.120706,0.00,44.248940
59,LODHA,31.582009,61.583012,0.00,37.266008
2,ADANIENT,6.108628,82.488756,0.00,35.438953
3,ADANIGREEN,16.941432,53.015873,0.00,27.982922
18,BHEL,8.119533,57.801418,0.00,26.368381
72,PNB,6.474121,58.731025,0.00,26.082059
17,BHARTIARTL,3.664440,61.439589,0.00,26.041611
87,TECHM,0.923148,57.488527,0.00,23.364670
28,DLF,8.931072,46.989721,0.00,22.368317


In [14]:
top_growth_companies.to_csv(
    OUTPUT / "top_growth_companies.csv",
    index=False
)

print("Top Growth Companies Saved Successfully!")

Top Growth Companies Saved Successfully!
